# Shakespeare's Characters

The goal of this notebook is to train a simple LSTM model **to mimic the dialogue of Shakespearean characters.** We will train the model on a large corpus of text from Shakespeare's plays, teaching it **to predict the next character in a sequence.** For example, given the input 'to be or no', the model should predict 't' as the most likely following character. The input window then slides forward by one position—now reading 'o be or not'—to predict the next token. This sliding window approach allows the model to generate text character-by-character continuously.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [2]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Data Loading

In [4]:
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'input.txt'

# 1. Download the file
urllib.request.urlretrieve(url, filename)

# 2. Read the text file
# Note: It's good practice to specify encoding='utf-8' on Windows to avoid UnicodeDecodeErrors
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Success! Loaded {len(text)} characters.")

Success! Loaded 1115394 characters.


In [15]:
# Create character-level vocabulary
chars = sorted(list(set(text)))

vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")
print(f"chars: {chars}")

stoi = {w: i for i, w in enumerate(chars)}
print("stoi:", stoi)
itos = {i: ch for i, ch in enumerate(chars)}
print("itos:", itos)

# Encode the entire text into integers
data = [stoi[c] for c in text]

Vocab size: 65
chars: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
stoi: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
itos: {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '

In [17]:
# Sliding Window Dataset
# Instead of sentences, we slice the giant text into overlapping sequences

class CharDataset(Dataset):
    def __init__(self, data, seq_len=100):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # x is a sequence of seq_len characters
        # y is the single character that comes right after it
        
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [33]:
# Parameters
seq_len = 100
batch_size = 128

# Create the dataset
dataset = CharDataset(data=data, seq_len=seq_len)

# Create the data loader
loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

## Model Definition

In [34]:
# Define the LSTM model
class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super().__init__()

        # Convert tokens into a vector of size 64 to feed them into the model
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Define LSTM block
        self.lstm = nn.LSTM(
            embed_size,         # length of embedded representation
            hidden_size,        # number of internal units to store information about the input sentence
            batch_first=True    # tells the model data comes in batches (in the form [batch_size, sequence, features])
        )

        # Fully Connected layer for final prediction
        # Outputs a probability for each word in the training set
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):

        emb = self.embedding(x)

        # Obtain the hidden state for every step in the sequence
        out, _ = self.lstm(emb)

        # Get the hidden state just for the last word
        last_hidden = out[:, -1, :]

        # Get probability for each word to be the next one
        logits = self.fc(last_hidden)

        return logits

In [30]:
model = LSTMLanguageModel(vocab_size).to(device)

##Training

In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [35]:
epochs = 5
model.train()

for epoch in range(epochs):
    total_loss = 0
    for i, (x, y) in enumerate(loader):
        # move x and y to correct device
        x = x.to(device)
        y = y.to(device)

        # clear out old gradients using optimizer
        optimizer.zero_grad()

        # compute the output for the model
        logits = model(x)

        # compute loss between logits and ground truth
        loss = criterion(logits, y)

        # compute how much each param contributes to the loss
        loss.backward()

        # adjust the parameters using the optimizer
        optimizer.step()

        total_loss += loss.item()

        if i % 500 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f} ---")

Epoch 1, Batch 0, Loss: 4.1781
Epoch 1, Batch 500, Loss: 1.9086
Epoch 1, Batch 1000, Loss: 2.0184
Epoch 1, Batch 1500, Loss: 1.9274
Epoch 1, Batch 2000, Loss: 1.6754
Epoch 1, Batch 2500, Loss: 1.8476
Epoch 1, Batch 3000, Loss: 1.6767
Epoch 1, Batch 3500, Loss: 1.7371
Epoch 1, Batch 4000, Loss: 1.5669
Epoch 1, Batch 4500, Loss: 1.3824
Epoch 1, Batch 5000, Loss: 1.7450
Epoch 1, Batch 5500, Loss: 1.6916
Epoch 1, Batch 6000, Loss: 1.7674
Epoch 1, Batch 6500, Loss: 1.6313
Epoch 1, Batch 7000, Loss: 1.5955
Epoch 1, Batch 7500, Loss: 1.5148
Epoch 1, Batch 8000, Loss: 1.6650
Epoch 1, Batch 8500, Loss: 1.4331
--- Epoch 1 Avg Loss: 1.7337 ---
Epoch 2, Batch 0, Loss: 1.6850
Epoch 2, Batch 500, Loss: 1.5057
Epoch 2, Batch 1000, Loss: 1.5697
Epoch 2, Batch 1500, Loss: 1.6644
Epoch 2, Batch 2000, Loss: 1.4756
Epoch 2, Batch 2500, Loss: 1.4172
Epoch 2, Batch 3000, Loss: 1.1960
Epoch 2, Batch 3500, Loss: 1.6262
Epoch 2, Batch 4000, Loss: 1.5100
Epoch 2, Batch 4500, Loss: 1.4844
Epoch 2, Batch 5000, Lo

## Validation

In [36]:
def generate_text(model, start_str="ROMEO: ", length=200, temperature=0.8):
    model.eval()
    generated = start_str

    for _ in range(length):
        # Format input string to length seq_len
        input_tokens = [stoi[c] for c in generated[-seq_len:]]
        x = torch.tensor(input_tokens).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

            # Apply temperature and sample (instead of argmax)
            # Higher temp = more "creative"/chaotic text
            probs = torch.softmax(logits / temperature, dim=-1)
            pred_idx = torch.multinomial(probs, num_samples=1).item()

            generated += itos[pred_idx]

    return generated

In [37]:
print("\nGenerated Shakespeare:\n")
print(generate_text(model, start_str="KING: ", length=300))


Generated Shakespeare:

KING: you
True, that it is the keppentured.

ROMEO:
But whom there were both to meth serve and pity.

ANTONIO:
I see fit us townath I safry, the should seef'd
That sad, sir, I was hear to come,
While me the tryselves of his seats of send
it's comn lies him, and an unishory prison,
I have be reput his unpl
